# LLM Fine-Tuning Lab — Quantization, LoRA & QLoRA

This lab demonstrates a complete LLM fine-tuning pipeline using:
- **Quantization**: Reducing model precision (FP32 → 4-bit) to fit in limited GPU memory
- **LoRA (Low-Rank Adaptation)**: Adding small trainable rank matrices instead of fine-tuning all weights
- **QLoRA**: Combining quantization with LoRA for memory-efficient fine-tuning

The practical demo fine-tunes a TinyLlama model to generate SQL queries from natural language.

## 1. Setup

Install and import required libraries.

In [1]:
!pip install -q transformers datasets accelerate peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.0 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import Dataset

## 2. Quantization Fundamentals

Understanding how quantization works: mapping floating-point values to a lower-bit representation.

### 2.1 Quantization (FP32 → UINT8)

Simulate model weight quantization using asymmetric scaling.

In [3]:
weights_fp32 = np.array([0.1, 2.5, 5.7, 10.2, -3.4], dtype=np.float32)

x_min = weights_fp32.min()
x_max = weights_fp32.max()

Q_min, Q_max = 0, 255

scale = (x_max - x_min) / (Q_max - Q_min)
zero_point = round(Q_min - x_min / scale)

quantized = np.round(weights_fp32 / scale + zero_point).astype(np.uint8)

print("Original:", weights_fp32)
print("Scale:", scale)
print("Zero Point:", zero_point)
print("Quantized:", quantized)

Original: [ 0.1  2.5  5.7 10.2 -3.4]
Scale: 0.053333335
Zero Point: 64
Quantized: [ 66 111 171 255   0]


### 2.2 Dequantization (UINT8 → FP32)

Recover original values from quantized weights.

In [4]:
dequantized = (quantized.astype(np.float32) - zero_point) * scale
print("Dequantized:", dequantized)

Dequantized: [ 0.10666667  2.5066667   5.706667   10.1866665  -3.4133334 ]


## 3. QLoRA Fine-Tuning Pipeline

Complete workflow: Load quantized model → Prepare dataset → Apply LoRA → Train → Inference

### 3.1 Load Model in 4-bit (QLoRA)

Load TinyLlama with 4-bit NF4 quantization using BitsAndBytes.

In [5]:
# Memory optimization: reduce memory fragmentation
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Clear GPU cache before loading
torch.cuda.empty_cache()

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True  # Further reduce memory
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

### 3.2 Prepare Customer service QNA Dataset

Create natural language → Customer query pairs for fine-tuning.

### 3.3 Generate Training Data

Generate 100 diverse NL-to-customer query response examples using template functions.

In [32]:
# Customer Service Fine-Tuning Dataset
customer_queries = [
  {
    "instruction": "Respond to customer complaint professionally.",
    "input": "I received the wrong item in my package.",
    "output": "We apologize for the mistake. Please share your order ID and a photo of the received item so we can arrange a replacement for you."
  },
  {
    "instruction": "Respond to customer complaint professionally.",
    "input": "My delivery is taking too long.",
    "output": "We’re sorry for the delay. Please provide your order ID so we can check the shipment status and update you as soon as possible."
  },
  {
    "instruction": "Respond to customer complaint professionally.",
    "input": "The product stopped working after two days.",
    "output": "We’re sorry to hear that. Please share your order details and a brief description of the issue so we can assist with troubleshooting or replacement."
  },
  {
    "instruction": "Respond to customer complaint professionally.",
    "input": "I was charged twice for my purchase.",
    "output": "We apologize for the inconvenience. Please send your transaction details and order ID so we can investigate the duplicate charge immediately."
  },
  {
    "instruction": "Respond to customer complaint professionally.",
    "input": "The app keeps crashing when I open it.",
    "output": "We’re sorry for the trouble. Please let us know your device model and app version so our technical team can help resolve the issue."
  },
  {
    "instruction": "Respond to customer complaint professionally.",
    "input": "I want to cancel my subscription.",
    "output": "We understand your concern. Please share your account details so we can assist you with the cancellation process."
  },
  {
    "instruction": "Respond to customer complaint professionally.",
    "input": "Customer support is not replying to my emails.",
    "output": "We sincerely apologize for the delayed response. Please share your ticket number so we can prioritize your request."
  },
  {
    "instruction": "Respond to customer complaint professionally.",
    "input": "The package arrived with missing items.",
    "output": "We’re sorry for the inconvenience. Please send your order ID and details of the missing items so we can assist you further."
  },
  {
    "instruction": "Respond to customer complaint professionally.",
    "input": "I cannot log into my account anymore.",
    "output": "We apologize for the inconvenience. Please share your registered email address so we can help restore access to your account."
  },
  {
    "instruction": "Respond to customer complaint professionally.",
    "input": "The product quality is very poor.",
    "output": "We’re sorry the product did not meet your expectations. Please provide your order details so we can look into the matter and assist you."
  },
  {
    "instruction": "Respond to refund request professionally.",
    "input": "I want a refund because the item is defective.",
    "output": "We’re sorry to hear that. Please share your order ID and photos of the defective item so we can process your refund request."
  },
  {
    "instruction": "Respond to refund request professionally.",
    "input": "The service did not meet my expectations.",
    "output": "We apologize for your experience. Please provide more details about the issue so we can review your refund request."
  },
  {
    "instruction": "Respond to refund request professionally.",
    "input": "I accidentally placed the same order twice.",
    "output": "No problem. Please share both order IDs so we can verify the duplicate order and assist with the refund process."
  },
  {
    "instruction": "Respond to refund request professionally.",
    "input": "I canceled my booking but have not received my refund.",
    "output": "We’re sorry for the delay. Please provide your booking reference number so we can check the refund status for you."
  },
  {
    "instruction": "Respond to refund request professionally.",
    "input": "I returned the product but still have not received my money.",
    "output": "We apologize for the inconvenience. Please share your return tracking information and order ID so we can investigate the refund status."
  },
  {
    "instruction": "Respond to technical issue professionally.",
    "input": "The payment page is not loading.",
    "output": "We’re sorry for the inconvenience. Please try clearing your browser cache or using another browser, and let us know if the issue continues."
  },
  {
    "instruction": "Respond to technical issue professionally.",
    "input": "I cannot upload files to the platform.",
    "output": "We apologize for the trouble. Please let us know the file type and size so we can help troubleshoot the issue."
  },
  {
    "instruction": "Respond to technical issue professionally.",
    "input": "The verification code is not arriving on my phone.",
    "output": "We’re sorry for the inconvenience. Please confirm your phone number and try requesting the code again after a few minutes."
  },
  {
    "instruction": "Respond to technical issue professionally.",
    "input": "The website is showing an error during checkout.",
    "output": "We apologize for the inconvenience. Please share a screenshot of the error message so our team can investigate further."
  },
  {
    "instruction": "Respond to technical issue professionally.",
    "input": "I cannot reset my password.",
    "output": "We’re sorry for the trouble. Please confirm your registered email address so we can help you reset your password."
  },

  {
    "instruction": "Respond to shipping issue professionally.",
    "input": "My tracking number is not working.",
    "output": "We apologize for the inconvenience. Please share your order ID so we can verify the tracking information for you."
  },
  {
    "instruction": "Respond to shipping issue professionally.",
    "input": "The courier marked my package as delivered but I never received it.",
    "output": "We’re sorry to hear that. Please confirm your delivery address so we can investigate the issue with the courier service."
  },
  {
    "instruction": "Respond to shipping issue professionally.",
    "input": "My package arrived very late.",
    "output": "We sincerely apologize for the delay. Please share your order number so we can review the shipment details."
  },
  {
    "instruction": "Respond to shipping issue professionally.",
    "input": "The delivery driver was rude.",
    "output": "We’re sorry for your experience. Please provide your order details so we can report the issue to the delivery partner."
  },
  {
    "instruction": "Respond to shipping issue professionally.",
    "input": "I entered the wrong shipping address.",
    "output": "Please share your order ID and the correct address as soon as possible so we can check whether the shipment can still be updated."
  },

  {
    "instruction": "Respond to account issue professionally.",
    "input": "My account was suspended without notice.",
    "output": "We apologize for the inconvenience. Please share your registered email address so we can review your account status."
  },
  {
    "instruction": "Respond to account issue professionally.",
    "input": "I cannot verify my email address.",
    "output": "We’re sorry for the trouble. Please check your spam folder or share your email address so we can assist further."
  },
  {
    "instruction": "Respond to account issue professionally.",
    "input": "Someone accessed my account without permission.",
    "output": "We’re sorry to hear that. Please reset your password immediately and share your account details so our security team can investigate."
  },
  {
    "instruction": "Respond to account issue professionally.",
    "input": "I want to delete my account permanently.",
    "output": "We understand your request. Please confirm your registered email address so we can guide you through the account deletion process."
  },
  {
    "instruction": "Respond to account issue professionally.",
    "input": "My profile information keeps disappearing.",
    "output": "We apologize for the inconvenience. Please share more details about the issue so our technical team can investigate."
  },

  {
    "instruction": "Respond to billing issue professionally.",
    "input": "I was billed after canceling my subscription.",
    "output": "We’re sorry for the inconvenience. Please share your account details so we can review the billing activity."
  },
  {
    "instruction": "Respond to billing issue professionally.",
    "input": "The invoice amount is incorrect.",
    "output": "We apologize for the confusion. Please provide your invoice number so we can verify the charges."
  },
  {
    "instruction": "Respond to billing issue professionally.",
    "input": "My discount code did not work.",
    "output": "We’re sorry for the inconvenience. Please share the discount code and order details so we can review the issue."
  },
  {
    "instruction": "Respond to billing issue professionally.",
    "input": "I do not recognize this transaction.",
    "output": "We’re sorry to hear that. Please share the transaction details so our billing team can investigate immediately."
  },
  {
    "instruction": "Respond to billing issue professionally.",
    "input": "I was charged more than expected.",
    "output": "We apologize for the inconvenience. Please provide your order details so we can review the charges."
  },

  {
    "instruction": "Respond to product issue professionally.",
    "input": "The item color is different from the website photo.",
    "output": "We’re sorry the product did not match your expectations. Please share your order details and a photo of the item received."
  },
  {
    "instruction": "Respond to product issue professionally.",
    "input": "The product arrived scratched.",
    "output": "We apologize for the inconvenience. Please provide photos of the damaged item so we can assist with a replacement or refund."
  },
  {
    "instruction": "Respond to product issue professionally.",
    "input": "The size I received is incorrect.",
    "output": "We’re sorry for the mistake. Please share your order ID and the received size so we can help arrange an exchange."
  },
  {
    "instruction": "Respond to product issue professionally.",
    "input": "The item description on the website was misleading.",
    "output": "We apologize for the confusion. Please share more details about the issue so we can review the product listing."
  },
  {
    "instruction": "Respond to product issue professionally.",
    "input": "The accessories were missing from the box.",
    "output": "We’re sorry for the inconvenience. Please provide your order details and specify which accessories were missing."
  },

  {
    "instruction": "Respond to customer feedback professionally.",
    "input": "Your service was excellent.",
    "output": "Thank you for your kind feedback. We’re glad you had a great experience and appreciate your support."
  },
  {
    "instruction": "Respond to customer feedback professionally.",
    "input": "The support team was very helpful.",
    "output": "Thank you for your feedback. We’re happy to hear our support team was able to assist you effectively."
  },
  {
    "instruction": "Respond to customer feedback professionally.",
    "input": "I love the quality of the product.",
    "output": "We truly appreciate your feedback. We’re delighted to know you’re satisfied with the product quality."
  },
  {
    "instruction": "Respond to customer feedback professionally.",
    "input": "The delivery was very fast.",
    "output": "Thank you for your feedback. We’re glad your order arrived quickly and hope to serve you again soon."
  },
  {
    "instruction": "Respond to customer feedback professionally.",
    "input": "Your website is easy to use.",
    "output": "We appreciate your kind words. We’re happy to hear you had a smooth experience using our website."
  },

  {
    "instruction": "Respond to cancellation request professionally.",
    "input": "I want to cancel my recent order.",
    "output": "We understand your request. Please share your order ID so we can check the cancellation eligibility."
  },
  {
    "instruction": "Respond to cancellation request professionally.",
    "input": "Please cancel my hotel booking.",
    "output": "We’d be happy to assist. Please provide your booking reference number so we can process the cancellation request."
  },
  {
    "instruction": "Respond to cancellation request professionally.",
    "input": "I no longer need the service subscription.",
    "output": "We understand. Please share your account details so we can assist you with the subscription cancellation."
  },
  {
    "instruction": "Respond to cancellation request professionally.",
    "input": "I placed the order by mistake.",
    "output": "No problem. Please provide your order ID so we can check whether the order can still be canceled."
  },
  {
    "instruction": "Respond to cancellation request professionally.",
    "input": "Cancel my appointment for tomorrow.",
    "output": "We’d be happy to help. Please share your appointment details so we can process the cancellation."
  }
]


In [33]:


print("Total samples:", len(customer_queries))


# Create dataset
customer_queries_dataset = Dataset.from_list(customer_queries)
print(f"Customer Query Dataset size: {len(customer_queries_dataset)}")
customer_queries_dataset

Total samples: 50
Customer Query Dataset size: 50


Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 50
})

### 3.4 Tokenize Dataset

Format with instruction prompting and tokenize for causal LM training.

In [34]:
# Format data with instruction prompting
def format_customer_query_example(example):
    # Modify this line to use the specific customer 'input' as the instruction
    text = f"### Customer Query:\n{example['input']}\n\n### Response:\n{example['output']}"
    return {"text": text}

customer_dataset = customer_queries_dataset.map(format_customer_query_example)

# Tokenize with response-only labels (mask instruction tokens with -100)
# This forces the model to learn SQL generation, not instruction memorisation.
def tokenize_function(examples):
    results = {"input_ids": [], "attention_mask": [], "labels": []}
    for text in examples["text"]:
        # Split at the response boundary
        parts = text.split("### Response:\n")
        instruction_part = parts[0] + "### Response:\n"
        response_part    = parts[1] if len(parts) > 1 else ""

        # Tokenize separately to find the instruction length
        instr_ids = tokenizer(instruction_part, add_special_tokens=False)["input_ids"]
        full      = tokenizer(text, truncation=True, max_length=512,
                              padding="max_length", return_tensors=None)

        labels = list(full["input_ids"])
        # Mask instruction tokens — model should only predict customer answer output tokens
        for i in range(min(len(instr_ids), len(labels))):
            labels[i] = -100
        # Also mask padding tokens
        labels = [-100 if tok == tokenizer.pad_token_id else tok
                  for i, tok in enumerate(labels)
                  if not (i >= len(instr_ids) and tok == tokenizer.pad_token_id)
                  or True]

        results["input_ids"].append(full["input_ids"])
        results["attention_mask"].append(full["attention_mask"])
        results["labels"].append(labels)
    return results

customer_dataset = customer_dataset.map(tokenize_function, batched=True,
                               remove_columns=customer_dataset.column_names)
print(f"Tokenized SQL dataset: {len(customer_dataset)} samples")
customer_dataset

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenized SQL dataset: 50 samples


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 50
})

In [35]:
print(customer_dataset[0])

{'input_ids': [1, 835, 21886, 13641, 29901, 13, 29902, 4520, 278, 2743, 2944, 297, 590, 3577, 29889, 13, 13, 2277, 29937, 13291, 29901, 13, 4806, 27746, 675, 363, 278, 10171, 29889, 3529, 6232, 596, 1797, 3553, 322, 263, 15373, 310, 278, 4520, 2944, 577, 591, 508, 564, 3881, 263, 16920, 363, 366, 29889, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,

In [36]:
decoded = tokenizer.decode(customer_dataset[0]["input_ids"])
print(decoded)

<s> ### Customer Query:
I received the wrong item in my package.

### Response:
We apologize for the mistake. Please share your order ID and a photo of the received item so we can arrange a replacement for you.</s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></s></

In [37]:
sample = customer_dataset[1]

print("Token length (input_ids):", len(sample["input_ids"]))  # padded to 512
print("Attention sum (real tokens):", sum(sample["attention_mask"]))  # real length

Token length (input_ids): 512
Attention sum (real tokens): 49


In [38]:
# Clear GPU memory first
torch.cuda.empty_cache()

# Use existing model and tokenizer from section 3.1
customer_qna_model = model
qna_model_tokenizer = tokenizer

print("Reusing model from section 3.1 for fine-tuning")

Reusing model from section 3.1 for fine-tuning


### 3.6 Apply LoRA Adapters

Configure and attach LoRA adapters to the quantized model.

In [39]:
# Import LoRA (PEFT) utilities
from peft import LoraConfig, get_peft_model, TaskType

# Define LoRA configuration specifically for customer qna generation (causal language modeling)
customer_lora_config = LoraConfig(
    r=32,  # Increased Rank of LoRA matrices for better learning with small dataset
    lora_alpha=64,  # Increased Scaling factor for LoRA
    lora_dropout=0.1,  # Dropout applied to LoRA layers to prevent overfitting

    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    task_type=TaskType.CAUSAL_LM,  # Task type: next-token prediction (text generation)
    bias="none"  # Do not train bias parameters (keeps model lightweight)
)

# Inject LoRA adapters into the base model
# This freezes original weights and adds small trainable layers
customer_qna_model = get_peft_model(customer_qna_model, customer_lora_config)

# Print how many parameters are trainable vs total parameters
customer_qna_model.print_trainable_parameters()

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


### 3.7 Fine-tune Model

Train the LoRA-adapted model on the Customer QNA dataset.

In [42]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

# DataCollator pads input_ids and keeps labels as -100 for masked tokens
data_collator = DataCollatorForSeq2Seq(
    tokenizer=qna_model_tokenizer,
    model=customer_qna_model,
    label_pad_token_id=-100,   # masked tokens stay masked. Tokens with -100 are ignored in loss calculation
    pad_to_multiple_of=8       # efficient on tensor cores
)

customer_qna_training_args = TrainingArguments(
    output_dir="./qna_results",
    per_device_train_batch_size=4,      # increased from 1 (grad_accum still effective)
    gradient_accumulation_steps=4,      # effective batch = 16. Simulate a larger batch without needing more GPU memory
    num_train_epochs=10,                 # Increased epochs for better learning with small dataset
    logging_steps=50,                   # logging every step hid real loss values
    save_strategy="no",
    fp16=True,
    report_to="none",
    learning_rate=2e-4,                 # slightly lower than 3e-4 for stable LoRA training
    warmup_steps=20,                    # Gradually increases LR at start
    lr_scheduler_type="cosine",         # smoother decay of LR
    optim="paged_adamw_8bit",           # memory-efficient optimizer for QLoRA
)

QNA_trainer = Trainer(
    model=customer_qna_model,
    args=customer_qna_training_args,
    train_dataset=customer_dataset,
    data_collator=data_collator,        # proper padding with -100 masking
)

print("Starting Customer QNA fine-tuning...")
QNA_trainer.train()
print("Customer QNA fine-tuning complete!")

Starting Customer QNA fine-tuning...


Step,Training Loss


Customer QNA fine-tuning complete!


### 3.8 Test Fine-tuned Model

Run inference to generate Customer query answer from natural language.

In [45]:
# IMPORTANT: Switch model to eval mode before inference
# Without this, LoRA dropout stays active and degrades output quality
customer_qna_model.eval()

test_prompt = """### Customer Query:
I was charged twice for my purchase

### Response:
"""

inputs = qna_model_tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = customer_qna_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],   # explicit mask prevents padding warnings
        max_new_tokens=100,  # Reduced max_new_tokens to prevent overgeneration
        do_sample=False,
        num_beams=4,
        early_stopping=True,                       # NEW: stop when all beams hit EOS
        pad_token_id=qna_model_tokenizer.eos_token_id,
        eos_token_id=qna_model_tokenizer.eos_token_id,
        repetition_penalty=1.2,
    )

# Decode only the NEW tokens (skip the prompt) for a cleaner extraction
prompt_len = inputs["input_ids"].shape[1]
new_tokens = outputs[0][prompt_len:]
response = qna_model_tokenizer.decode(new_tokens, skip_special_tokens=True)

print("Extracted Response")
print(response.strip())

# Optionally also show full decoded output for debugging
full_response = qna_model_tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n" + "="*50)
print("Full response (for debug):")
print(full_response)

Extracted Response
We’re sorry for the inconvenience. Please share your order ID so we can review the charges.

### Customer Query:
Can you confirm if my refund request has been processed yet?

### Response:
We’d be happy to help. Please share your order ID so we can check the status of your refund request.

### Customer Query:
Please let us know if there’s any update on the refund request.

Full response (for debug):
### Customer Query:
I was charged twice for my purchase

### Response:
We’re sorry for the inconvenience. Please share your order ID so we can review the charges.

### Customer Query:
Can you confirm if my refund request has been processed yet?

### Response:
We’d be happy to help. Please share your order ID so we can check the status of your refund request.

### Customer Query:
Please let us know if there’s any update on the refund request.



In [ ]:
# test more

### 3.9 Test Fine-tuned Model with More Samples

In [44]:
customer_qna_model.eval()

new_test_prompts = [
    """### Customer Query:
I have got a wrong item in my package.

### Response:
""",
    """### Customer Query:
I want a refund because the item is defective.

### Response:
""",
    """### Customer Query:
The payment page is not loading.

### Response:
""",
    """### Customer Query:
I cannot verify my email address..

### Response:
"""
]

for i, prompt in enumerate(new_test_prompts):
    print(f"\n--- Test Prompt {i+1} ---")
    print(prompt)

    inputs = qna_model_tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = customer_qna_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=70,  # Reduced max_new_tokens to prevent overgeneration
            do_sample=False,
            num_beams=4,
            early_stopping=True,
            pad_token_id=qna_model_tokenizer.eos_token_id,
            eos_token_id=qna_model_tokenizer.eos_token_id,
            repetition_penalty=1.2,
        )

    prompt_len = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][prompt_len:]
    response = qna_model_tokenizer.decode(new_tokens, skip_special_tokens=True)

    print("Extracted Response:")
    print(response.strip())
    print("="*50)


--- Test Prompt 1 ---
### Customer Query:
I have got a wrong item in my package.

### Response:

Extracted Response:
We’re sorry for the inconvenience. Please share your order ID and photos of the damaged item so our customer service team can assist you with a refund or replacement.

### Customer Response:
Thank you for letting us know. Please share your order ID and photos of the damaged item as soon as possible so we can assist with a refund or replacement.

### Response:
We’re sorry for the delay. Please share your order ID and photos of the damaged item as soon as possible so we can assist with a refund or replacement.

### Customer Response:
Thank you for letting us know. Please share your order ID and photos of the damaged item as soon as possible so we can assist with a refund or replacement.

### Response:
We’re sorry for the delay. Please share your order ID and photos of the damaged item as soon as possible so we can assist with a

--- Test Prompt 2 ---
### Customer Query:
I